In [1]:
!pip install pymupdf sentence-transformers


[notice] A new release of pip is available: 26.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import fitz
import os
import json
from sentence_transformers import SentenceTransformer


load pdf


In [3]:
pdf_path = "digitalbook.pdf"
doc=fitz.open(pdf_path)
print ("Total pages:", len(doc))


Total pages: 565


In [4]:
pages = []
for page_number,page in enumerate(doc):
    text = page.get_text()
    pages.append({"page_number": page_number, "text": text})    
print(pages[0])  # Print the first page's content


{'page_number': 0, 'text': ''}


TEXT CLEANING

In [5]:
def clean_text(text):
    # Remove newlines and extra spaces
     text = text.replace('\n', ' ')
     text = ' '.join(text.split())
   
     return text

In [6]:
for page in pages:
    page['text'] = clean_text(page['text'])

print(pages[4])  # Print the cleaned first page's content




{'page_number': 4, 'text': 'Digital Design With an Introduction to the Verilog HDL FIFTH EDITION M. Morris Mano Emeritus Professor of Computer Engineering California State University, Los Angeles Michael D. Ciletti Emeritus Professor of Electrical and Computer Engineering University of Colorado at Colorado Springs Upper Saddle River Boston Columbus San Franciso New York Indianapolis London Toronto Sydney Singapore Tokyo Montreal Dubai Madrid Hong Kong Mexico City Munich Paris Amsterdam Cape Town'}


CHUNKING

In [7]:
def chunk_code(text, chunk_size=2000, overlap=200):
    """
    Splits the text into chunks of specified size with a specified overlap.
    
    Parameters:
    - text (str): The input text to be chunked.
    - chunk_size (int): The maximum size of each chunk.
    - overlap (int): The number of overlapping characters between chunks.
    
    Returns:
    - List[str]: A list of text chunks.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start += chunk_size - overlap  # Move start forward by chunk_size minus overlap
    return chunks


In [ ]:
chunks = []
for page in pages:
    page_chunks = chunk_code(page['text'])
    for chunk in page_chunks:
        chunks.append({"page_number": page['page_number'], "chunk": chunk})

print("tottal chunks:", len(chunks))
print(chunks[3])  # Print the first chunk's content

EMBEDDING

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
texts = [chunk['chunk'] for chunk in chunks]
embeddings = model.encode(texts,show_progress_bar=True)


In [11]:
print("Embeddings shape:", embeddings.shape)  # Print the shape of the embeddings


Embeddings shape: (906, 384)


In [12]:
for i, chunk in enumerate(chunks):
    chunk['embedding'] = embeddings[i].tolist()  # Convert numpy array to list for JSON serialization

SAVING THE PROCESSED DATA

In [13]:
with open('chunks_with_embeddings.json', 'w') as f:
    json.dump(chunks, f, indent=4)

STORE EMBEDDING IN VECTORED DATABASE

In [14]:
!pip install faiss-cpu


[notice] A new release of pip is available: 26.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import faiss
import numpy as np

In [16]:
embedding_matrix = np.array([chunk['embedding'] for chunk in chunks]).astype('float32')
print("Embedding matrix shape:", embedding_matrix.shape)  # Print the shape of the embedding matrix

Embedding matrix shape: (906, 384)


In [17]:
dim = embedding_matrix.shape[1]  # Dimension of the embeddings
index = faiss.IndexFlatL2(dim)  # Create a FAISS index for L2 distance
index.add(embedding_matrix)  # Add the embeddings to the index  
print("Number of vectors in the index:", index.ntotal)  # Print the number of vectors in the index

Number of vectors in the index: 906


SEMANTIC SEARCH


In [18]:
def semantic_search(query, top_k=5):
    """
    Performs semantic search on the indexed embeddings using the provided query.
    
    Parameters:
    - query (str): The input query string.
    - top_k (int): The number of top results to return.
    
    Returns:
    - List[Dict]: A list of dictionaries containing the page number, chunk, and distance for each result.
    """
    query_embedding = model.encode([query]).astype('float32')  # Encode the query
    distances, indices = index.search(query_embedding, top_k)  # Search the index
    results = []
    for i in range(top_k):
        idx = indices[0][i]
        distance = distances[0][i]
        results.append({
            "page_number": chunks[idx]['page_number'],
            "chunk": chunks[idx]['chunk'],
            "distance": float(distance)
        })
    return results    

In [ ]:
from sentence_transformers import SentenceTransformer

test_query = "What is the main topic of the book?"
# Ensure the embedding model is available before running semantic search
if 'model' not in globals():
    model = SentenceTransformer('all-MiniLM-L6-v2')

if 'index' not in globals():
    if 'embedding_matrix' not in globals():
        embedding_matrix = np.asarray(
            [chunk['embedding'] for chunk in chunks],
            dtype='float32'
        )
    index = faiss.IndexFlatL2(embedding_matrix.shape[1])
    index.add(embedding_matrix)

results = semantic_search(test_query, top_k=5)
for result in results:
    print(f"Page Number: {result['page_number']}, Distance: {result['distance']}")
    print(f"Chunk: {result['chunk']}\n")    

In [20]:
!pip install rank-bm25


[notice] A new release of pip is available: 26.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
from rank_bm25 import BM25Okapi

In [ ]:
corpus = [chunk['chunk'] for chunk in chunks]
bm25 = BM25Okapi([doc.split() for doc in corpus])
query = "What is the main topic of the book?"
scores = bm25.get_scores(query.split())
top_indices = scores.argsort()[-5:][::-1]  # Get indices of top 5 scores in descending order
for idx in top_indices:
    print(f"Page Number: {chunks[idx]['page_number']}, Score: {scores[idx]}")
    print(f"Chunk: {chunks[idx]['chunk']}\n")   

In [ ]:
# Map semantic results back to their chunk indices
semantic_rank = {}
for rank, item in enumerate(results, start=1):
    idx = next(
        i for i, chunk in enumerate(chunks)
        if chunk["page_number"] == item["page_number"]
        and chunk["chunk"] == item["chunk"]
    )
    semantic_rank[idx] = rank

bm25_rank = {int(idx): rank for rank, idx in enumerate(top_indices, start=1)}

# Reciprocal rank fusion; lower ranks are better
candidate_indices = set(semantic_rank) | set(bm25_rank)
final_scores = {
    idx: 1 / (60 + semantic_rank.get(idx, 1000))
       + 1 / (60 + bm25_rank.get(idx, 1000))
    for idx in candidate_indices
}

for idx in sorted(candidate_indices, key=final_scores.get, reverse=True)[:5]:
    print(f"Page Number: {chunks[idx]['page_number']}")
    print(f"Chunk: {chunks[idx]['chunk']}\n")

print("Final Scores:", final_scores)  # Print the final scores for debugging

In [ ]:
from sentence_transformers import cross_encoder

# Initialize the cross-encoder for semantic rankingre
reranker = cross_encoder.CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

In [25]:
!pip install ollama
import ollama


[notice] A new release of pip is available: 26.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
response = ollama.chat(
    model="llama3.1:8b",
    messages=[
        {
            "role": "user",
            "content": "Explain RAG in simple terms."
        }
    ]
)

print(response["message"]["content"])

In [ ]:
import ollama

response = ollama.chat(
    model="llama3.1:8b",
    messages=[
        {
            "role": "user",
            "content": "In AI, what does RAG stand for? Explain Retrieval-Augmented Generation in 2 simple sentences."
        }
    ]
)

print(response["message"]["content"])

In [ ]:
candidate_order = sorted(
    candidate_indices,
    key=final_scores.get,
    reverse=True
)[:10]

print("Number of candidates:", len(candidate_order))

pairs = [
    (test_query, chunks[idx]["chunk"])
    for idx in candidate_order
]

print("Pairs created:", len(pairs))

rerank_scores = reranker.predict(pairs)

reranked_results = sorted(
    zip(candidate_order, rerank_scores),
    key=lambda item: item[1],
    reverse=True
)

print("Reranking complete.")

for idx, score in reranked_results[:5]:
    print("Page:", chunks[idx]["page_number"])
    print("Reranker score:", float(score))
    print("Text:", chunks[idx]["chunk"][:500])
    print("-" * 80)

In [ ]:
def ask_documind(question, retrieval_k=5):

    # 1. Semantic search
    semantic_results = semantic_search(
        question,
        top_k=retrieval_k
    )

    semantic_rank = {}

    for rank, item in enumerate(semantic_results, start=1):
        idx = next(
            i for i, chunk in enumerate(chunks)
            if chunk["page_number"] == item["page_number"]
            and chunk["chunk"] == item["chunk"]
        )

        semantic_rank[idx] = rank

    # 2. BM25 keyword search
    tokenized_query = question.lower().split()

    bm25_scores = bm25.get_scores(tokenized_query)

    bm25_indices = bm25_scores.argsort()[-retrieval_k:][::-1]

    bm25_rank = {
        int(idx): rank
        for rank, idx in enumerate(bm25_indices, start=1)
    }

    # 3. Reciprocal Rank Fusion
    candidate_indices = set(semantic_rank) | set(bm25_rank)

    final_scores = {
        idx: (
            1 / (60 + semantic_rank.get(idx, 1000))
            + 1 / (60 + bm25_rank.get(idx, 1000))
        )
        for idx in candidate_indices
    }

    candidate_order = sorted(
        candidate_indices,
        key=final_scores.get,
        reverse=True
    )[:10]

    # 4. Neural reranking
    pairs = [
        (question, chunks[idx]["chunk"])
        for idx in candidate_order
    ]

    rerank_scores = reranker.predict(pairs)

    reranked_results = sorted(
        zip(candidate_order, rerank_scores),
        key=lambda item: item[1],
        reverse=True
    )

    # 5. Select the top 5 chunks
    top_results = reranked_results[:5]

    # 6. Build context with page metadata
    context_parts = []

    for idx, score in top_results:
        chunk = chunks[idx]

        context_parts.append(
            f"[Page {chunk['page_number']}]\n"
            f"{chunk['chunk']}"
        )

    context = "\n\n".join(context_parts)

    # 7. Create grounded prompt
    prompt = f"""
You are DocuMind, an AI document intelligence assistant.

Answer the user's question using ONLY the information provided in the context below.

If the answer is present in the context, answer it directly and clearly.
Do not say that the information is unavailable if the context contains the answer.
If the context genuinely does not contain the answer, then say:
"Information not found in the provided document."

DOCUMENT CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

    # 8. Generate answer using Ollama
    response = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "num_ctx": 4096,
            "num_predict": 400,
            "temperature": 0.1
        },
        keep_alive="5m"
    )

    answer = response["message"]["content"]

    return {
        "question": question,
        "answer": answer,
        "retrieved_chunks": [
            {
                "chunk_id": idx,
                "page_number": chunks[idx]["page_number"],
                "reranker_score": float(score),
                "text": chunks[idx]["chunk"]
            }
            for idx, score in top_results
        ]
    }

In [ ]:
result = ask_documind(
    "How does a D flip-flop work?"
)

print("QUESTION:")
print(result["question"])

print("\nANSWER:")
print(result["answer"])

print("\nSOURCES:")
for source in result["retrieved_chunks"]:
    print(
        f"Page: {source['page_number']} | "
        f"Reranker score: {source['reranker_score']:.4f}"
    )

SUMMARISATION

In [46]:
import ollama 
import re
def summarise_chunk_batch(batch, summary_type="detailed"):
    context_parts = []
    for chunk in batch:
        page_number = chunk.get("page_label")
        if page_number is None:
            page_number = chunk.get("page_number", "unknown")
            if isinstance(page_number, int):
                page_number += 1
        page_text = chunk.get("chunk", "")
        context_parts.append(f"Page {page_number}:\n{page_text}")
    context="\n\n".join(context_parts)
    summary_instruction={ "short": "Summarize the following text in a concise manner, highlighting the key points and main ideas.",
    "detailed": "Provide a detailed summary of the following text, including all relevant information and important details.",
    "bullet points": "Summarize the following text in bullet points, focusing on the most important aspects and key takeaways."
    }
    normalized_summary_type = summary_type.lower().replace(" summary", "")
    instruction=summary_instruction.get(normalized_summary_type, "Provide a detailed summary of the following text.")
    prompt=f"""You are DocuMind, an AI document intelligence assistant.Summarize the supplied document excerpts.

    Instructions:
    - {instruction}
    - Use only information in the supplied text.
    - Do not invent facts or introduce outside information.
- Preserve important technical terms.
- Include PDF page references for important claims.
- If the excerpts contain insufficient information, state that clearly.
- Avoid repeating the same idea.

DOCUMENT EXCERPTS:
{context}

SUMMARY:
"""
      
    response=ollama.chat(
        model="llama3.1:8b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "num_ctx": 1024,
            "num_batch": 64,
            "num_gpu": 0,
            "num_predict": 250,
            "temperature": 0.1
        },
        keep_alive="5m"
    )
    summary=response["message"]["content"]
    return summary   


In [47]:
def summarize_document(
    chunks,
    summary_type="detailed",
    batch_size=5,
    max_chunks=10
):
    """
    Summarize a document using map-reduce.

    max_chunks:
        10   = test with the first 10 chunks
        None = process all chunks
    """

    if not chunks:
        raise ValueError("No document chunks are available.")

    if batch_size < 1:
        raise ValueError("batch_size must be at least 1.")

    # Use a limited number of chunks during initial testing
    selected_chunks = (
        chunks[:max_chunks]
        if max_chunks is not None
        else chunks
    )

    intermediate_summaries = []

    # MAP: Summarize each batch
    for start in range(0, len(selected_chunks), batch_size):

        batch = selected_chunks[start:start + batch_size]

        start_page = batch[0].get(
            "page_label",
            batch[0]["page_number"] + 1
        )

        end_page = batch[-1].get(
            "page_label",
            batch[-1]["page_number"] + 1
        )

        print(
            f"Summarizing batch "
            f"{start // batch_size + 1} "
            f"(approximately pages {start_page}-{end_page})..."
        )

        batch_summary = summarise_chunk_batch(
            batch,
            summary_type=summary_type
        )

        intermediate_summaries.append({
            "start_page": start_page,
            "end_page": end_page,
            "summary": batch_summary
        })

    # REDUCE: Combine the intermediate summaries
    combined_summaries = "\n\n".join(
        (
            f"Source pages {item['start_page']}-"
            f"{item['end_page']}:\n"
            f"{item['summary']}"
        )
        for item in intermediate_summaries
    )

    final_prompt = f"""
You are DocuMind, an AI document summarization assistant.

Combine the following intermediate summaries into one coherent summary.

Requirements:
- {summary_type.replace("_", " ").capitalize()} summary.
- Preserve the main ideas and important technical details.
- Remove repetition.
- Do not introduce facts that are absent from the intermediate summaries.
- Retain PDF page references associated with the claims.
- If a claim cannot be traced to the supplied summaries, omit it.

INTERMEDIATE SUMMARIES:
{combined_summaries}

FINAL SUMMARY:
"""

    response = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {
                "role": "user",
                "content": final_prompt
            }
        ],
        options={
            "num_ctx": 1024,
            "num_batch": 64,
            "num_gpu": 0,
            "temperature": 0.1
        },
        keep_alive="5m"
    )

    return {
        "summary": response["message"]["content"].strip(),
        "chunks_processed": len(selected_chunks),
        "intermediate_summaries": intermediate_summaries
    }

OCR

In [ ]:
!pip install easyocr


In [ ]:
import easyocr
reader = easyocr.Reader(['en'], gpu=False)
def extract_text_from_pdf(pdf_path):
    pdf_doc = fitz.open(pdf_path)
    text = ""
    for pdf_page in pdf_doc:
        text += pdf_page.get_text()
    return text

def ocr_page(pdf_page, zoom=2):
    """
    Perform OCR on a single page of a PDF.

    Parameters:
    - page: A PyMuPDF page object.
    - zoom: The zoom factor for rendering the page image.

    Returns:
    - str: The extracted text from the page.
    """
    mat = fitz.Matrix(zoom, zoom)  # Create a transformation matrix for zooming
    pix = pdf_page.get_pixmap(matrix=mat)  # Render the page to a pixmap
    img = pix.tobytes("png")  # Convert the pixmap to PNG bytes
    result = reader.readtext(img)  # Perform OCR on the image
    ocr_text = " ".join([item[1] for item in result])  # Concatenate recognized text
    return ocr_text

ocr_pages = []
for page_number, pdf_page in enumerate(doc):
    ocr_text = ocr_page(pdf_page)  # Perform OCR on the page
    ocr_pages.append({"page_number": page_number, "text": ocr_text})  # Store the OCR text with page number
    

In [ ]:
# Filter out malformed entries before summarizing
safe_chunks = [
    chunk for chunk in chunks
    if isinstance(chunk, dict)
    and isinstance(chunk.get("chunk"), str)
    and chunk["chunk"].strip()
]

if not safe_chunks:
    print("No valid chunks available for summarization.")
else:
    summary_result = summarize_document(
        chunks=safe_chunks,
        summary_type="detailed",
        batch_size=2,
        max_chunks=min(4, len(safe_chunks)),
    )
    print("Final Summary:")
    print(summary_result["summary"])
print("Final Summary:")
print(summary_result["summary"])

In [ ]:
import psutil

memory = psutil.virtual_memory()

print(f"Total RAM: {memory.total / (1024**3):.2f} GB")
print(f"Available RAM: {memory.available / (1024**3):.2f} GB")
print(f"RAM Used: {memory.percent}%")

MULTIDOCUMENT SUPPORT

In [ ]:
import json
import hashlib
from pathlib import Path

import fitz  # pip install pymupdf
from sentence_transformers import SentenceTransformer

# Storage
INDEX_FILE = Path("chunks_with_embeddings.json")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


def load_chunks():
    if INDEX_FILE.exists():
        with open(INDEX_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return []


def save_chunks(chunks):
    with open(INDEX_FILE, "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False)


def split_text(text, chunk_size=900, overlap=150):
    text = " ".join(text.split())
    chunks = []

    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end == len(text):
            break

        start = end - overlap

    return chunks


def add_documents(pdf_files):
    """
    pdf_files: list of PDF file paths.
    Index multiple PDFs without duplicating previously indexed files.
    """
    indexed_chunks = load_chunks()
    existing_ids = {
        chunk.get("document_id")
        for chunk in indexed_chunks
        if chunk.get("document_id")
    }

    for pdf_path in pdf_files:
        pdf_path = Path(pdf_path)
        pdf_bytes = pdf_path.read_bytes()
        document_id = hashlib.sha256(pdf_bytes).hexdigest()

        if document_id in existing_ids:
            print(f"Already indexed: {pdf_path.name}")
            continue

        new_chunks = []

        with fitz.open(stream=pdf_bytes, filetype="pdf") as pdf:
            for page_number, page in enumerate(pdf, start=1):
                text = page.get_text("text")

                for chunk_text in split_text(text):
                    new_chunks.append({
                        "document_id": document_id,
                        "document_name": pdf_path.name,
                        "page": page_number,
                        "text": chunk_text,
                    })

        if not new_chunks:
            print(f"No extractable text: {pdf_path.name}")
            continue

        embeddings = embedding_model.encode(
            [chunk["text"] for chunk in new_chunks],
            normalize_embeddings=True
        )

        for chunk, embedding in zip(new_chunks, embeddings):
            chunk["embedding"] = embedding.tolist()

        indexed_chunks.extend(new_chunks)
        existing_ids.add(document_id)

        print(f"Indexed {pdf_path.name}: {len(new_chunks)} chunks")

    save_chunks(indexed_chunks)
    return indexed_chunks


def list_documents():
    """List all uniquely indexed PDFs."""
    chunks = load_chunks()
    documents = {}

    for chunk in chunks:
        doc_id = chunk.get("document_id")
        if doc_id:
            documents[doc_id] = {
                "document_id": doc_id,
                "filename": chunk["document_name"],
            }

    return list(documents.values())


# Example:
if __name__ == "__main__":
    add_documents([
        "document1.pdf",
        "document2.pdf",
        "document3.pdf",
    ])

    print(list_documents())